# 04.5 — Testing and consolidation

[Class index and setup](README.md) · [Previous: 04.4](04.4-public-data-and-authentication.ipynb)

**Purpose:** Leave evidence that the client handles bad input and explain the AI previews honestly.

**In class:** 0 minutes. **Independent work:** 75 minutes, included in the index's 210-minute allowance. Preparation has its own 30 minutes. The instructor's separate project is additional work.

## Learning outcomes

- Test normal results and at least two meaningful failure behaviors offline.
- Replace external acquisition while keeping decode/validation real.
- Distinguish a saved AI envelope from a live model result.
- Hand over an explained client and reproducible checks.

## Starting point

You know basic Python, functions, JSON-like dictionaries, and the earlier Observation/Report idea. The complete package and fixtures are supplied again; no previous notebook or exercise answer is needed. Open the supplied class workspace in VS Code, select this class's `.venv`, then **Restart Kernel → Run All**.

Default cells inspect saved responses and run offline checks. They do not start a server, call an external API, read credentials, install software, or download models. Live commands are explicit **manual terminal steps**. An HTTP-shaped fixture is not evidence of a live request. Empty answer cells are safe.

Attempt exercises before using the separate `%load` cell: uncomment its last line and run to reveal, then run the populated cell separately. A loaded solution replaces its loader cell; restarting the kernel does not restore that source. Keep your attempt in its own cell.

In [ ]:
from pathlib import Path
import sys

if not Path("report_app/api.py").is_file() or not Path("fixtures/observations.json").is_file():
    raise FileNotFoundError(
        "Open 04-consuming-apis.code-workspace and restart the kernel. "
        "Download the complete class folder, including report_app/, fixtures/, and solutions/."
    )
print("Notebook kernel:", sys.executable)
print("Class assets available; default execution is offline.")

## Independent work · 75 minutes — Test what can break

Allocate **45 minutes** to Exercise 04.5.H1 and **30 minutes** to Exercise 04.5.H2. Reading, executing checks, explanation, and the completion review are inside those allocations.

A real service is a poor required test dependency: it may be offline, rate-limited, revised, or inaccessible. Replace the external function the client calls, then execute the real status/JSON/schema path. Keep a separate live smoke test as evidence of integration, not a condition of deterministic CI.

Read [tests/test_http.py](tests/test_http.py). Its `replace_get` fixture installs a controlled GET. It returns a real Requests response or raises a transport exception. The tests still use the real parser. A response with HTTP 404 and invalid JSON must report the HTTP problem first.

| Test | Wrong behavior it would catch |
| --- | --- |
| Known normal summary | Client loses or misreads valid records |
| Timeout | Boundary leaks an unexplained transport failure |
| HTTP status before body | Error HTML is mistakenly diagnosed as invalid successful JSON |
| Invalid JSON | Program assumes every response body is JSON |
| Missing value in one row | Program accepts partial/broken records |
| Empty server filter | Program treats valid no-match data as a failed request |
| Missing public population | Program invents a zero measurement |

The server-function check calls the supplied route directly as ordinary Python. It checks filtering but does not send HTTP or test URL routing. The instructor verifies real localhost HTTP separately. Requests is the only HTTP client needed in this class.

In [ ]:
import subprocess
import sys

result = subprocess.run(
    [sys.executable, "-m", "pytest", "-q"],
    capture_output=True, text=True, timeout=60,
)
print(result.stdout)
if result.returncode:
    print(result.stderr)
assert result.returncode == 0, "The supplied offline suite should pass; inspect the actual failure above."

### 🧩 Exercise 04.5.H1 — Write two failure tests

> **Independent work · 45 minutes**
>
> **Goal:** Show that expected failures stay visible without depending on a live service.
>
> **Start here:** The supplied tests, saved_response helper, pytest.raises, and fetch_observations.
>
> **Task:** In a scratch copy write a timeout test and a missing-value test. Replace GET, keep the real parser, and assert useful failure messages. Run the tests. Then temporarily remove the relevant handling or validation in that scratch copy and show that a test fails; restore it and rerun.
>
> **Check:** Both tests pass with the intended implementation and at least one fails for the deliberate regression. A collection/import error does not count as detecting the regression. No network calls occur.
>
> **Hint:** Use patch(..., side_effect=requests.Timeout()) for transport and saved_response("invalid-record.json") for bad records. Do not replace parse_observations with a prevalidated answer.

In [ ]:
# Your attempt for 04.5.H1.
# Record code, observations, and your explanation here.

In [ ]:
# Reveal only after attempting the exercise.
# Remove the leading '# ' from the next line and run this cell.
# %load solutions/exercise_04_5_h1.py

### Read AI JSON without assuming both formats are equal

For this short review, read the illustrative envelopes already supplied. An OpenAI Responses envelope includes status and an output collection. An Ollama generate envelope exposes response and done. The transport pattern is similar; the contracts differ.

The tiny reference only extracts text from the **known completed fixture**. A production application must also handle refusals, incomplete output, non-message items, and API errors. Class 05 will cover application-level orchestration, validation, and evaluation. Do not mistake our fixture's sentence for a measured live-model result.

### 🧩 Exercise 04.5.H2 — Explain the AI request and response

> **Independent work · 30 minutes**
>
> **Goal:** Recognize the API pattern without adding accounts, SDKs, or model setup.
>
> **Start here:** examples/ai_http.py and the two illustrative AI JSON fixtures.
>
> **Task:** Run both default replay commands from AI_DEMOS.md. Identify method, URL, JSON body, timeout, and any authorization header in the live source. Extract text from both known saved envelopes; explain their structural difference. Write one sentence distinguishing HTTP success, schema validity, and factual correctness.
>
> **Check:** You can explain both POST examples and locate their saved text. You label the evidence illustrative/offline, keep credentials out of the notebook, and do not claim a live model evaluation.
>
> **Hint:** OpenAI text is inside message content items of type output_text; Ollama text is in response. Never print the bearer header.

In [ ]:
# Your attempt for 04.5.H2.
# Record code, observations, and your explanation here.

In [ ]:
# Reveal only after attempting the exercise.
# Remove the leading '# ' from the next line and run this cell.
# %load solutions/exercise_04_5_h2.py

## Required completion check — included in the two exercises

Show a request with an explicit timeout, validated observations, a useful failure, two deterministic failure tests, and an explanation of frontend/backend/API roles. Show the public-data mapping and whether a live request was possible. Explain the OpenAI/Ollama HTTP examples using saved data. A peer should be able to follow the class README without earlier exercise answers.

AI assistants may help, but you must explain the boundary, modify a small case, and show a test that catches an actual mistake. There is no new assessed-project rubric here.

## Optional extension — up to 180 minutes, excluded

Pick only the work you have time for. These are not required completion criteria.

- **Pagination · 45 minutes:** inspect [saved page one](fixtures/page-1.json) and [saved page two](fixtures/page-2.json). These are a synthetic teaching envelope, not World Bank responses. Follow next_page until null, collect three observations exactly once, and reject a repeated page or a supplied maximum-page limit. Explain why stopping after page one silently loses data. Adapt a real provider only after reading its own pagination contract.
- **Bounded retries · 45 minutes:** in a scratch copy make a supplied sequence “503, then 200” succeed within at most three GET attempts. Test the attempt limit with controlled responses and a replaced sleep. Retry only suitable read-only requests and selected temporary failures, respect Retry-After/service guidance, and stop on invalid JSON/schema or unauthorized access. Do not automatically retry a potentially charged model POST after a timeout: the server may already have processed it.
- **Extend FastAPI · 45 minutes:** add a separate /summary route in a scratch copy using the existing calculation; check empty/nonempty behavior. Consume it with Requests. No authentication server or deployment required.
- **Live AI observation · 45 minutes:** only with prepared instructor-approved access or an already-running local model, use the explicit live switch, record status/shape and inspect the result. Installation/downloads also consume these optional minutes; defer if they do not fit. Do not claim the chosen local model is CPU-tested unless you actually test it.

## References

- [pytest monkeypatch](https://docs.pytest.org/en/stable/how-to/monkeypatch.html)
- [unittest.mock patch](https://docs.python.org/3/library/unittest.mock.html#patch)
- [FastAPI first steps](https://fastapi.tiangolo.com/tutorial/first-steps/)
- [Instructor verification](INSTRUCTOR.md)
- [Class index](README.md)